# Build BUSCO Gene Based Phylogenetic Trees for Members of the Same Species (Hypocreaceae)

In [ ]:
## Import dependencies

import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tarfile

## Load BUSCOs and Remove Low BUSCO Percentage Assemblies

#### First, load list of Hypocreaceae accessions and identify this subset of BUSCO results zip files

In [ ]:
## Define the assemblies for the proposed BUSCO tree

SameSpecies = ["GCA_000167675.2", "GCA_002006585.1",
               "GCA_052818895.1", "GCA_047716485.1",
               "GCA_047716445.1", "GCA_047716435.1",
               "GCA_047716415.1", "GCA_042257965.1",
               "GCA_038428105.1", "GCA_028871515.1",
               "GCA_016806875.1", "GCA_028871095.1",
               "GCA_016806815.1", "GCA_028870775.1"]
SameSpeciesNames = ["Trichoderma reesei", "Trichoderma reesei",
                    "Trichoderma reesei", "Trichoderma reesei",
                    "Trichoderma reesei", "Trichoderma reesei",
                    "Trichoderma reesei", "Trichoderma reesei",
                    "Trichoderma reesei", "Trichoderma reesei",
                    "Trichoderma reesei", "Trichoderma reesei",
                    "Trichoderma reesei", "Trichoderma reesei"]


In [ ]:
# Build DataFrame
Hypocreaceae_df = pd.DataFrame({
    "Assembly Accession": SameSpecies,
    "Species": SameSpeciesNames
})

# Extract genus
Hypocreaceae_df["Genus"] = Hypocreaceae_df["Species"].apply(lambda x: x.split()[0])

In [ ]:
Hypocreaceae_df

In [ ]:
## See how many assemblies there are for a particular genus and related genera 

'''
# Count the occurrences of each unique genus and create a DataFrame
Genus_counts = Hypocreaceae_df['Genus'].value_counts().reset_index()
Genus_counts.columns = ['Genus', 'Count']  # Rename columns

# Create a horizontal bar chart
plt.figure(figsize=(10, 9))
bar_plot = sns.barplot(x='Count', y='Genus', data=Genus_counts)

# Annotate the bars with counts
for p in bar_plot.patches:
    bar_plot.annotate(f'{int(p.get_width())}',  # Text to display
                      (p.get_width() + 0.5, p.get_y() + 0.2 + p.get_height()/2),  # Position
                       fontsize=15, color='black')

plt.xlabel('Count')
plt.ylabel('Genus')
plt.title('Number of Assemblies in Genbank')
plt.show()
'''

In [ ]:
## Make list of the zip folders that correspond to selected taxa BUSCO results

# Switch to directory containing zip files of BUSCO results
os.chdir("/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/Hypocreaceae/BUSCO_Hypocreaceae/busco_results")

# Extract accessions from zip file folder names for cross-referencing with list of selected Hypocreaceae accessions
AllfungiBUSCOs_ZipFiles = os.listdir(os.getcwd())
AllfungiBUSCOs_ZipAccessions = ["_".join(x.split("_")[3:]) for x in AllfungiBUSCOs_ZipFiles]
AllfungiBUSCOs_ZipAccessions = [".".join(x.split(".")[:2]) for x in AllfungiBUSCOs_ZipAccessions]

# Make list of Hypocreaceae accessions from dataframe containing Hypocreaceae GenBank information
HypocreaceaeList = list(Hypocreaceae_df["Assembly Accession"])

# Find indices of elements in list of fungi zip files that are also in list of selected taxa
SelectedBUSCOs_ZipAccessionIndices = [i for i, x in enumerate(AllfungiBUSCOs_ZipAccessions) if x in HypocreaceaeList]

# Use indices determined above to extract filenames of selected taxa BUSCO results as new list
SelectedBUSCOs_ZipFiles = [AllfungiBUSCOs_ZipFiles[i] for i in SelectedBUSCOs_ZipAccessionIndices]

In [ ]:
## Sanity check that the correct number of zip files are listed

print(len(HypocreaceaeList))
print(len(SelectedBUSCOs_ZipFiles))

In [ ]:
HypocreaceaeList

In [ ]:
## Second sanity check: see if files look like the correct taxa under consideration

#SelectedBUSCOs_ZipFiles

In [ ]:
## Determine the completeness and single copy number directly from BUSCO results compressed tarfiles

# Path to the tar.gz file
file_path = '/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/Hypocreaceae/BUSCO_Hypocreaceae/busco_results/'

# Create lists for containing results of computations
Completeness = []
SingleCopies = []
Duplicates = []
Fragments = []
Missing = []
count = 0     # Initialize counter used in loop

for i in np.arange(len(SelectedBUSCOs_ZipFiles)):
    count += 1     # Count number of files operated on
    
    # Tarfile operations
    with tarfile.open(file_path+SelectedBUSCOs_ZipFiles[i], 'r:gz') as tar:
    
        # Read the contents of the tar file
        fileList = tar.getnames()      # get filenames for content of tar file
        print("Running for file: ", fileList[0], "     File#"+str(count))     # Print for intermittent status update
        member = tar.getmember(fileList[0]+'/short_summary.txt')     # Select file of interest
        file_content = tar.extractfile(member).read()    # Read file
        lines = file_content.decode('utf-8').split('\n')     # Decode from bytes to readable file, and delimit new lines

        # Compute completeness and number of single copy BUSCO genes 
        #(the following delimits by tabs and selects the appropriate numbers from the BUSCO summary text file)
        NumberComplete = int(lines[9].split('\t')[1])
        NumberSingleCopy = int(lines[10].split('\t')[1])
        NumberDuplicate = int(lines[11].split('\t')[1])
        NumberFragment = int(lines[12].split('\t')[1])
        NumberMissing = int(lines[13].split('\t')[1])
        NumberTotal = int(lines[14].split('\t')[1])
        PercentComplete = 100*(NumberComplete/NumberTotal)
        # Append results to appropriate lists
        Completeness.append(PercentComplete)
        SingleCopies.append(NumberSingleCopy)
        Duplicates.append(NumberDuplicate)
        Fragments.append(NumberFragment)
        Missing.append(NumberMissing)

In [ ]:
## No completeness/duplicate-count filtering is applied in this analysis; all selected assemblies are used as-is.
Over80Complete = SelectedBUSCOs_ZipFiles

print("Total assemblies used: ", len(Over80Complete), " of ", len(SelectedBUSCOs_ZipFiles))


In [ ]:
Over80Complete

In [ ]:
# Create directory, then switch to it
os.makedirs('/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/Hypocreaceae/BUSCO_Hypocreaceae/SameSpecies', exist_ok=True)
os.chdir('/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/Hypocreaceae/BUSCO_Hypocreaceae/SameSpecies')

# Create a DataFrame
data = {
    'SingleCopies': SingleCopies,
    'Duplicates': Duplicates,
    'Fragments': Fragments,
    'Missing': Missing,
    'Completeness': Completeness,
    'File' : SelectedBUSCOs_ZipFiles
}

df = pd.DataFrame(data)

# Save to a CSV file
output_file = "Hypocreaceae_BUSCO_Results.csv"
df.to_csv(output_file, index=False)

print(f"DataFrame saved to {output_file}")

In [ ]:
for i in np.arange(len(SingleCopies)):
    print("S:", SingleCopies[i],"; D:", Duplicates[i], "; F:", Fragments[i], "; M:", Missing[i], ";  C:", Completeness[i])

In [ ]:
import io 

## Unzip and collect the names of all single copy busco .faa files

# Path to the tar.gz files
file_path = '/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/Hypocreaceae/BUSCO_Hypocreaceae/busco_results/'

# Create lists for containing results of computations
AAseqFiles = []
count = 0     # Initialize counter used in loop

for i in np.arange(len(Over80Complete)):
    count += 1     # Count number of files operated on
    print(f"Processing file {count}: {Over80Complete[i]}")
    
    # Open the main tar.gz file
    with tarfile.open(file_path + Over80Complete[i], 'r:gz') as tar:
        
        # Extract the single_copy_busco_sequences.tar.gz from the main tar
        single_copy_tar_path = 'run_fungi_odb10/busco_sequences/single_copy_busco_sequences.tar.gz'
        
        try:
            # Get the single_copy_busco_sequences.tar.gz file
            single_copy_tar_member = tar.getmember(single_copy_tar_path)
            single_copy_tar_content = tar.extractfile(single_copy_tar_member).read()
            
            # Now open the inner tar.gz file
            with tarfile.open(fileobj=io.BytesIO(single_copy_tar_content), mode='r:gz') as inner_tar:
                # List the contents of the inner tar file
                inner_file_list = inner_tar.getnames()
                
                # Get all .faa files from the inner tar
                faa_files = [file for file in inner_file_list if file.endswith(".faa")]
                AAseqFiles.append(faa_files)
                
                print(f"  Found {len(faa_files)} single copy BUSCO .faa files")
                
        except KeyError:
            print(f"  ERROR: Could not find {single_copy_tar_path} in archive")
            AAseqFiles.append([])  # Add empty list for consistency

print(f"\nTotal files processed: {count}")

In [ ]:
## Determine the common single copy BUSCOs across all considered sequences
## Find the intersection of all faa file lists

import os
import sys
from pathlib import Path
import numpy as np

print("=== Calculating Common BUSCO Elements ===")

# Check if AAseqFiles was successfully defined
if 'AAseqFiles' not in locals() or not AAseqFiles:
    print("[FATAL] AAseqFiles is empty or undefined. Cannot find common BUSCOs.")
    print("Please run the previous cell that extracts AA sequence files from BUSCO results.")
    CommonElements = []
else:
    # Make a copy of the list of lists of .faa files for iterating over
    AAseqFilesClean = AAseqFiles.copy()

    # Robustly remove the path information and keep only the filename (BUSCO ID)
    for i in range(len(AAseqFiles)):
        AAseqFilesClean[i] = [Path(x).name for x in AAseqFiles[i]]

    # Start with the first list as a set
    if AAseqFilesClean:
        common_buscos = set(AAseqFilesClean[0])
        
        # Find intersection with all other lists
        for file_list in AAseqFilesClean[1:]:
            common_buscos.intersection_update(file_list)
        
        CommonElements = sorted(list(common_buscos))
        
        print(f"✓ Found {len(CommonElements)} common single copy BUSCOs across {len(AAseqFilesClean)} assemblies")
        
        if CommonElements:
            print(f"Sample BUSCO IDs: {CommonElements[:5]}...")
        else:
            print("WARNING: No common BUSCOs found! Check if assemblies are too divergent.")
    else:
        print("ERROR: AAseqFilesClean is empty")
        CommonElements = []

# Sanity check
if not CommonElements:
    print("\n❌ CRITICAL: No common BUSCOs found. Pipeline cannot continue.")
    print("Possible reasons:")
    print("  - Assemblies are too phylogenetically distant")
    print("  - BUSCO runs failed for some assemblies")
    print("  - Different BUSCO versions used")
else:
    print(f"\n✅ SUCCESS: {len(CommonElements)} common BUSCOs ready for alignment")



In [ ]:

import io
import subprocess
import sys
import time
import re
import shutil
import os 
import collections
from pathlib import Path
import numpy as np

## --- SLURM Helper Functions ---
def submit_job(command):
    """Submit an sbatch job and return the job ID."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    print(result.stdout)
    match = re.search(r'Submitted batch job (\d+)', result.stdout)
    if not match:
        raise RuntimeError(f"Failed to submit job: {result.stdout}\n{result.stderr}")
    return match.group(1)

def wait_for_job(job_id, poll_interval=30):
    """Wait until the SLURM job finishes."""
    while True:
        result = subprocess.run(["squeue", "-j", str(job_id)], capture_output=True, text=True)
        if job_id not in result.stdout:
            print(f"Job {job_id} finished.")
            break
        else:
            print(f"Job {job_id} still running... waiting {poll_interval}s")
            time.sleep(poll_interval)
## --- End SLURM Helper Functions ---

# --- Robust FASTA Parsing Function ---
def _robust_fasta_parser(filepath):
    """Parses a FASTA file and returns a dictionary of {header: sequence}."""
    sequences = collections.OrderedDict()
    current_header = None
    current_seq_lines = []
    
    try:
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue 

                if line.startswith('>'):
                    if current_header is not None:
                        sequences[current_header] = "".join(current_seq_lines)
                    
                    # CRITICAL: Extract only the accession ID that will be used as the key.
                    current_header = line.lstrip('>').split('\n')[0].split()[0]
                    current_seq_lines = []
                elif current_header is not None:
                    current_seq_lines.append(line)
        
        # Save the very last sequence
        if current_header is not None:
            sequences[current_header] = "".join(current_seq_lines)

    except Exception as e:
        print(f"Error reading FASTA file {filepath}: {e}")
        return collections.OrderedDict()
        
    return sequences

In [ ]:
print("=== Setting up directories and files for alignment pipeline ===\n")

## --- Define File Paths and Directories ---
family_name = 'Hypocreaceae' 
experiment_type = 'SameSpecies' 

# Use the variables defined in the preceding cells: CommonElements and Over80Complete
if 'CommonElements' not in locals() or not CommonElements:
    print("[FATAL] CommonElements is empty or undefined. No common BUSCOs shared across all selected assemblies.")
    sys.exit(1)

if 'Over80Complete' not in locals() or not Over80Complete:
    print("[FATAL] Over80Complete is empty or undefined.")
    sys.exit(1)

# Determine stable base directory
BASE_DIR_NAME = f'{experiment_type}_pipeline'
NOTEBOOK_CWD = Path.cwd()
BASE_DIR = NOTEBOOK_CWD / BASE_DIR_NAME

TRIM_DIR = BASE_DIR / f'Trimmed_Alignments_{experiment_type}'
TEMP_FASTA_DIR = BASE_DIR / 'Temp_Gene_Fastas'
SLURM_LOG_DIR = BASE_DIR / 'slurm_logs'

# Ensure directories are created in the new, stable location
BASE_DIR.mkdir(exist_ok=True)
TRIM_DIR.mkdir(exist_ok=True)
TEMP_FASTA_DIR.mkdir(exist_ok=True)
SLURM_LOG_DIR.mkdir(exist_ok=True)
os.chdir(BASE_DIR)

print(f"Working directory: {BASE_DIR}")
print(f"Trim directory: {TRIM_DIR}")
print(f"Temp FASTA directory: {TEMP_FASTA_DIR}")
print(f"SLURM log directory: {SLURM_LOG_DIR}")

# File names for inter-script communication
COMMON_BUSCOS_LIST = 'CommonBUSCOs.txt'
COMPLETE_ASSEMBLIES_LIST = 'Over80CompleteAssemblies.txt'

# Save the list of common BUSCOs (one per line) for the shell script to read
common_buscos_path = BASE_DIR / COMMON_BUSCOS_LIST
with open(common_buscos_path, 'w') as f:
    f.write('\n'.join(CommonElements) + '\n')
print(f"✓ Saved {len(CommonElements)} common BUSCOs to {common_buscos_path}")

# Save the list of complete assembly IDs (folder names) for the shell script to read
complete_assemblies_path = BASE_DIR / COMPLETE_ASSEMBLIES_LIST
with open(complete_assemblies_path, 'w') as f:
    f.write('\n'.join(Over80Complete) + '\n')
print(f"✓ Saved {len(Over80Complete)} assembly files to {complete_assemblies_path}")

total_genes = len(CommonElements)
print(f"\n📊 Pipeline Summary:")
print(f"   Total genes to process in parallel: {total_genes}")
print(f"   Total assemblies used in supermatrix: {len(Over80Complete)}")

In [ ]:
print("=== Creating alignment and trimming script ===\n")

# Create the alignment script directly in Python instead of modifying an external file
alignment_script_content = '''#!/bin/bash
#SBATCH --job-name=Busco_AT_%A_%a
#SBATCH --output=slurm_logs/busco_AT_%a.out
#SBATCH --error=slurm_logs/busco_AT_%a.err
#SBATCH --cpus-per-task=1  
#SBATCH --mem=2G
#SBATCH --partition=YOUR_PARTITION   # set this to your cluster's partition/account
#SBATCH --array=1-{total_genes}%50   

# --- Variables passed from Python Script ---
FAMILY=$1
EXP_TYPE=$2
BUSCO_RESULTS_DIR=$3
BASE_DIR=$4

# Derived paths
TRIM_DIR="$BASE_DIR/Trimmed_Alignments_$EXP_TYPE"
TEMP_FASTA_DIR="$BASE_DIR/Temp_Gene_Fastas"

# Read the list of common BUSCOs (one per line) and select the gene for this array task
BUSCO_GENE_LIST="$BASE_DIR/CommonBUSCOs.txt"
BUSCO_GENE=$(sed -n "${{SLURM_ARRAY_TASK_ID}}p" "$BUSCO_GENE_LIST")

if [ -z "$BUSCO_GENE" ]; then
    echo "Error: Could not find BUSCO gene for array task ID $SLURM_ARRAY_TASK_ID."
    exit 1
fi

echo "--- Processing Gene: $BUSCO_GENE (Task ID: $SLURM_ARRAY_TASK_ID) ---"

# --- 1. Extract Sequences into a single temporary FASTA file ---
gene_fasta_temp="$TEMP_FASTA_DIR/$BUSCO_GENE.faa"
> "$gene_fasta_temp" # Create/clear the temp file

# Extract the base BUSCO ID by removing the final '.faa' suffix.
BUSCO_ID=$(echo "$BUSCO_GENE" | sed 's/\\.faa$//')

# DEBUG: Count the assemblies we're processing
echo "Processing assemblies from: $BASE_DIR/Over80CompleteAssemblies.txt"
assembly_count=$(wc -l < "$BASE_DIR/Over80CompleteAssemblies.txt")
echo "Total assemblies to process: $assembly_count"

# Read the list of tar.gz files to process
while IFS= read -r tar_file_name; do
    # Skip empty lines
    [ -z "$tar_file_name" ] && continue
    
    # Extract the full accession (e.g., GCA_030784555.1) from the filename
    # More robust pattern matching
    accession=$(echo "$tar_file_name" | sed -E 's/^run_fungi_odb10_(.+\\.(GCA_[^.]+\\.\\d+))\\.tar\\.gz$/\\2/' | head -1)
    
    # If the above fails, try a simpler approach
    if [ "$accession" = "$tar_file_name" ]; then
        accession=$(echo "$tar_file_name" | sed 's/^run_fungi_odb10_//' | sed 's/\\.tar\\.gz$//')
    fi
    
    echo "  Processing assembly: $accession (from $tar_file_name)"

    # Path to the inner single_copy_busco_sequences.tar.gz
    SINGLE_COPY_TAR_PATH='run_fungi_odb10/busco_sequences/single_copy_busco_sequences.tar.gz'
    FAA_MEMBER_PATH="single_copy_busco_sequences/$BUSCO_ID.faa" 

    # Use tar command to extract content
    SEQUENCE=$(tar --to-stdout -xzf "$BUSCO_RESULTS_DIR/$tar_file_name" "$SINGLE_COPY_TAR_PATH" 2>/dev/null \\
    | tar --to-stdout -xzf - "$FAA_MEMBER_PATH" 2>/dev/null \\
    | grep -v '^>' \\
    | tr -d '\\n')

    # Ensure the sequence is not empty before writing
    if [ -n "$SEQUENCE" ]; then
        # Use printf to reliably write the header (>accession) followed by a newline
        printf ">%s\\n%s\\n" "$accession" "$SEQUENCE" >> "$gene_fasta_temp"
        echo "    ✓ Successfully extracted sequence for $accession"
    else
        # Log the missing sequence
        echo "    WARNING: Assembly $accession is MISSING the gene $BUSCO_GENE"
    fi

done < "$BASE_DIR/Over80CompleteAssemblies.txt"

# Count how many sequences we actually extracted
extracted_count=$(grep -c '^>' "$gene_fasta_temp" 2>/dev/null || echo 0)
echo "Total sequences extracted: $extracted_count"

# --- 2. Align the gene sequences using MAFFT ---
aligned_fasta_temp="$TEMP_FASTA_DIR/$BUSCO_GENE.aln"
echo "Starting MAFFT alignment..."

# Check if the temporary FASTA file is empty (meaning no sequences were extracted)
if [ ! -s "$gene_fasta_temp" ]; then
    echo "ERROR: No sequences found for $BUSCO_GENE. Skipping alignment and trimming."
    exit 0
fi

mafft --auto --maxiterate 1000 --thread 1 "$gene_fasta_temp" > "$aligned_fasta_temp" 2> /dev/null

if [ $? -ne 0 ]; then
    echo "MAFFT failed for $BUSCO_GENE. Skipping."
    exit 0 
fi

# --- 3. Trim the alignment using TrimAl ---
trimmed_fasta_out="$TRIM_DIR/$BUSCO_GENE.trimmed.faa"
echo "Starting TrimAl trimming..."
trimal -in "$aligned_fasta_temp" -out "$trimmed_fasta_out" -strictplus

if [ $? -ne 0 ]; then
    echo "TrimAl failed for $BUSCO_GENE. Skipping."
fi

# --- Cleanup ---
rm -f "$gene_fasta_temp" "$aligned_fasta_temp"
echo "Finished $BUSCO_GENE"
'''

# Write the script to a file
alignment_script_path = BASE_DIR / 'busco_align_trim.sh'
with open(alignment_script_path, 'w') as f:
    f.write(alignment_script_content.format(total_genes=total_genes))

# Make the script executable
subprocess.run(['chmod', '+x', str(alignment_script_path)], check=True)

print(f"✓ Created alignment script: {alignment_script_path}")
print(f"✓ Script configured for {total_genes} genes")

In [ ]:
print("=== Submitting alignment and trimming jobs ===\n")

# The shell script needs the 'file_path' which defines the root BUSCO directory
busco_results_root = Path('/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/') / family_name / "BUSCO_Hypocreaceae" / "busco_results"

if not busco_results_root.exists():
    print(f"❌ ERROR: BUSCO results directory not found: {busco_results_root}")
    sys.exit(1)

print(f"BUSCO results directory: {busco_results_root}")

# Submit the SLURM array job
align_trim_cmd = (
    f'sbatch '
    f'--job-name={family_name}_{experiment_type}_ATarray '
    f'{alignment_script_path} '
    f'{family_name} {experiment_type} {busco_results_root} {BASE_DIR}'
)

print(f"Submitting {total_genes} alignment/trimming tasks...")
print(f"Command: {align_trim_cmd}")

try:
    align_trim_job = submit_job(align_trim_cmd)
    print(f"✓ Successfully submitted job {align_trim_job}")
    print("Waiting for job completion...")
    wait_for_job(align_trim_job)
    print("✓ Alignment and trimming completed successfully!")
except Exception as e:
    print(f"❌ Failed to submit alignment job: {e}")
    sys.exit(1)

In [ ]:
print("=== Concatenating trimmed alignments into supermatrix ===\n")

Supermatrix_Out_File = f'{family_name}_{experiment_type}_BUSCOsTrimmed_Supermatrix.fa'
Partition_Out_File = f'{family_name}_{experiment_type}_BUSCOsTrimmed_partitions.txt'

# --- Read and Organize Sequences ---
assembly_sequences = {} 
partition_data = []
current_start = 1

# List the trimmed alignment files
trimmed_files = sorted([f for f in TRIM_DIR.iterdir() if f.name.endswith('.trimmed.faa')])
if not trimmed_files:
    print(f"❌ ERROR: No trimmed alignment files found in {TRIM_DIR}")
    print("This usually means the SLURM array job failed.")
    sys.exit(1)

print(f"Found {len(trimmed_files)} trimmed alignment files")

# Pre-populate assembly_sequences with all required accessions initialized to empty strings
expected_assemblies = [filename.split('_')[3] + '_' + filename.split('_')[4].replace('.tar.gz', '') for filename in Over80Complete]
for accession_key in expected_assemblies:
    assembly_sequences[accession_key] = ''

print(f"Processing {len(trimmed_files)} gene alignments...")

for filepath in trimmed_files:
    # Gene ID is the filename without the suffix
    gene_id = filepath.name.replace('.trimmed.faa', '')
    if gene_id not in CommonElements:
        print(f"Warning: Trimmed file {filepath.name} is not in CommonElements. Skipping.")
        continue 
        
    gene_length = 0
    
    # Use the robust parser
    gene_sequences = _robust_fasta_parser(filepath)

    if gene_sequences:
        # Get the length of the first sequence read
        first_seq = list(gene_sequences.values())[0] if gene_sequences else ""
        gene_length = len(first_seq)
    
    if gene_length == 0:
        print(f"WARNING: Skipping alignment file {filepath.name} - Length is zero.")
        continue

    # Update partition data
    partition_data.append(f"WAG, {gene_id} = {current_start}-{current_start + gene_length - 1}")
    current_start += gene_length
    
    # Concatenate sequence for each expected assembly, padding with gaps if missing
    for accession_key in expected_assemblies: 
        # Get the sequence or a gap string if the assembly is missing the gene
        seq = gene_sequences.get(accession_key, '-' * gene_length)
        
        # Ensure sequence length is correct (sanity check)
        if len(seq) != gene_length:
            print(f"WARNING: Length mismatch for {accession_key} in gene {gene_id}. Expected: {gene_length}, Found: {len(seq)}. Padding/Truncating.")
            if len(seq) > gene_length:
                seq = seq[:gene_length]
            else:
                seq = seq + ('-' * (gene_length - len(seq)))
            
        assembly_sequences[accession_key] += seq

print(f"✓ Processed all gene alignments")
print(f"Total supermatrix length: {current_start - 1} amino acids")

In [ ]:
print("=== Writing supermatrix and partition files ===\n")

# Write Supermatrix
try:
    with open(Supermatrix_Out_File, 'w') as f_supermatrix:
        for assembly_id, sequence in assembly_sequences.items():
            # Write accession ID as the FASTA header
            f_supermatrix.write(f">{assembly_id}\n{sequence}\n")
    print(f"✓ Supermatrix saved to {Supermatrix_Out_File}")
except Exception as e:
    print(f"❌ Failed to write supermatrix: {e}")
    sys.exit(1)

# Write Partition File
try:
    with open(Partition_Out_File, 'w') as f_partition:
        f_partition.write('\n'.join(partition_data) + '\n')
    print(f"✓ Partition file saved to {Partition_Out_File}")
except Exception as e:
    print(f"❌ Failed to write partition file: {e}")
    sys.exit(1)

# Final checks
if os.path.exists(Supermatrix_Out_File) and os.path.getsize(Supermatrix_Out_File) > 0:
    total_length = current_start - 1
    if total_length == 0:
        print("❌ ERROR: Calculated total alignment length is zero!")
        sys.exit(1)
        
    print(f"✅ Concatenation successful!")
    print(f"   Supermatrix: {Supermatrix_Out_File}")
    print(f"   Partition: {Partition_Out_File}")
    print(f"   Total length: {total_length} amino acids")
    print(f"   Number of assemblies: {len(assembly_sequences)}")
    
else:
    print("❌ ERROR: Python concatenation failed. Supermatrix file is missing or empty.")
    sys.exit(1)

# Clean up TEMP_FASTA_DIR 
shutil.rmtree(TEMP_FASTA_DIR, ignore_errors=True)
print(f"✓ Cleaned up temporary directory: {TEMP_FASTA_DIR}")

# Save the supermatrix filename for the IQ-Tree step
FinalAlignmentFile = Supermatrix_Out_File
print(f"✓ Final alignment file ready for IQ-Tree: {FinalAlignmentFile}")

In [ ]:
from pathlib import Path

# --- Set your original partition file ---
Partition_Out_File = "Hypocreaceae_SameSpecies_BUSCOsTrimmed_partitions.txt"
corrected_partition_file = Partition_Out_File.replace('.txt', '_iqtree_spp_ready.txt')

# --- Read original partition file ---
with open(Partition_Out_File, 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

# --- Filter lines that contain '=' and strip any unwanted whitespace ---
clean_lines = []
for line in lines:
    if '=' in line:
        # Remove leading/trailing whitespace around commas and '='
        parts = line.split('=')
        left, right = parts[0].strip(), parts[1].strip()
        
        # Keep model and gene name intact (everything before '=')
        clean_lines.append(f"{left} = {right}")

# --- Save corrected file compatible with -spp ---
with open(corrected_partition_file, 'w') as f:
    f.write('\n'.join(clean_lines))

print(f"✓ Partition file rebuilt and ready for -spp: {corrected_partition_file}")

In [ ]:
print("=== Running IQ-Tree with absolute paths ===\n")

# Cancel any previous failed jobs
subprocess.run(f"scancel -n {family_name}_{experiment_type}_iqtree", shell=True, capture_output=True)

# Use absolute paths to ensure files are found
tree_cmd = (
    f'sbatch --wrap="cd {BASE_DIR} && iqtree2 '
    f'-s {FinalAlignmentFile} '
    f'-spp {corrected_partition_file} '
    f'-T AUTO -m MFP+MERGE -bb 1000 -alrt 1000 --redo" '
    f'--job-name={family_name}_{experiment_type}_iqtree '
    f'--output={family_name}_{experiment_type}_iqtreeOutput_%j.out '
    f'--error={family_name}_{experiment_type}_iqtreeError_%j.err '
    '--cpus-per-task=32 --mem=32G --partition=YOUR_PARTITION'
)

print(f"Command: {tree_cmd}")

try:
    tree_job = submit_job(tree_cmd)
    print(f"✓ Successfully submitted IQ-Tree job {tree_job}")
    print("Waiting for job completion...")
    wait_for_job(tree_job)
    print("✅ IQ-Tree job completed successfully!")
    print("All jobs completed successfully.")
except Exception as e:
    print(f"❌ Failed: {e}")

In [ ]:
# Save this variable as it is used again later, don't want to have to rerun code to obtain variable
with open("Hypocreaceae_SameSpecies_Over80Complete.txt", "w") as file:
    for item in Over80Complete:
        file.write(item + "\n")

In [ ]:
os.chdir('/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/Hypocreaceae/BUSCO_Hypocreaceae/SameSpecies/SameSpecies_pipeline')
with open("Hypocreaceae_SameSpecies_Over80Complete.txt", "r") as file:
    Over80Complete = [line.strip() for line in file]

In [ ]:
## Display maximum likelihood tree determined using IQtree2


from Bio import Phylo



# Load the IQ-TREE output file (convert to Newick format if necessary)
tree = Phylo.read("/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/Hypocreaceae/BUSCO_Hypocreaceae/SameSpecies/SameSpecies_pipeline/Hypocreaceae_SameSpecies_BUSCOsTrimmed_partitions_iqtree_spp_ready.txt.treefile", "newick")


# Plot the tree
fig = plt.figure(figsize=(10, 12))
ax = fig.add_subplot(1, 1, 1)
Phylo.draw(tree, axes=ax)
plt.show()


In [ ]:
Over80Complete

In [ ]:
Over80Accessions = [filename.split('_')[3] + '_' + filename.split('_')[4].replace('.tar.gz', '') for filename in Over80Complete]


Over80Accessions

In [ ]:
## Create a dictionary which related accession numbers to organism names

# Dictionary comprehension to extract appropriate information
Hypocreaceae_AccessionToSpecies = {
    accession: f"{Hypocreaceae_df.loc[Hypocreaceae_df['Assembly Accession'] == accession, 'Species'].values[0]} ({accession})"
    for accession in Over80Accessions
    if accession in Hypocreaceae_df['Assembly Accession'].values
}

# Display the resulting dictionary
print(Hypocreaceae_AccessionToSpecies)

In [ ]:
import pickle


with open('HypocreaceaeDictionary.pkl', 'wb') as file:  # Note 'wb' for writing
    pickle.dump(Hypocreaceae_AccessionToSpecies, file)

In [ ]:
import pickle


# Load dictionary from a file
with open('HypocreaceaeDictionary.pkl', 'rb') as file:
    Hypocreaceae_AccessionToSpecies = pickle.load(file)

In [ ]:

# Load the IQ-TREE output file (Newick format)
tree = Phylo.read("/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/Kmers_SpeciesGenusFamily/Hypocreaceae/BUSCO_Hypocreaceae/SameSpecies/SameSpecies_pipeline/Hypocreaceae_SameSpecies_BUSCOsTrimmed_partitions_iqtree_spp_ready.txt.treefile", "newick")

tree.ladderize()

# Update the leaf labels in the tree
for leaf in tree.get_terminals():
    if leaf.name in Hypocreaceae_AccessionToSpecies:
        leaf.name = Hypocreaceae_AccessionToSpecies[leaf.name]

# Plot the updated tree
fig = plt.figure(figsize=(10, 15))
ax = fig.add_subplot(1, 1, 1)
Phylo.draw(tree, axes=ax)
plt.show()
